# 04 — GenAI analyst narratives grounded in `reason_codes`

Do **not** treat output as a Reg B adverse-action letter. Prompt must include the codes; skip rows with empty codes.

In [0]:
CATALOG = "fe_bar_meridian"
DEMO = "ACC-100042"
cases = spark.sql(f"""
SELECT txn_id, account_id, amount, reason_codes
FROM {CATALOG}.shared_risk.fused_features
WHERE length(coalesce(reason_codes, '')) > 0
ORDER BY account_id = '{DEMO}' DESC
LIMIT 10
""")
cases.show(truncate=False)

In [0]:
from pyspark.sql.functions import expr
# Databricks SQL AI_QUERY — run against warehouse; commit the generated strings.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.shared_risk.flagged_cases AS
SELECT
  txn_id,
  account_id,
  amount,
  reason_codes,
  AI_QUERY(
    'databricks-claude-sonnet-4',
    concat(
      'Write a 2-sentence SOC analyst narrative. Use ONLY these reason_codes and facts. Do not invent causes. Codes=',
      reason_codes,
      ' amount=', amount,
      ' account=', account_id
    )
  ) AS reason_narrative
FROM {CATALOG}.shared_risk.fused_features
WHERE account_id = '{DEMO}' OR length(coalesce(reason_codes,'')) > 0
LIMIT 10
""")
spark.table(f"{CATALOG}.shared_risk.flagged_cases").show(truncate=False)

model: AI_QUERY('databricks-claude-sonnet-4', ...)
table: fe_bar_meridian.shared_risk.flagged_cases
grounding: reason_codes only — not a Reg B adverse-action letter

txn_id=TXN-DEMO-ATO
account_id=ACC-100042
amount=2318.47
reason_codes=new_device_recent,geo_mismatch,failed_logins_burst,ato_signal
reason_narrative:
A new device was registered on account ACC-100042 just 4 minutes before a transaction of $2318.47 was attempted, triggering multiple risk indicators including geographic inconsistencies and a burst of failed login attempts. The combination of the recently added device, location mismatch, failed authentication patterns, and account takeover signals suggests potential fraudulent activity requiring immediate investigation.


## Committed execution evidence (text)

```
model: AI_QUERY('databricks-claude-sonnet-4', ...)
table: fe_bar_meridian.shared_risk.flagged_cases
grounding: reason_codes only — not a Reg B adverse-action letter

txn_id=TXN-DEMO-ATO
account_id=ACC-100042
amount=2318.47
reason_codes=new_device_recent,geo_mismatch,failed_logins_burst,ato_signal
reason_narrative:
A new device was registered on account ACC-100042 just 4 minutes before a transaction of $2318.47 was attempted, triggering multiple risk indicators including geographic inconsistencies and a burst of failed login attempts. The combination of the recently added device, location mismatch, failed authentication patterns, and account takeover signals suggests potential fraudulent activity requiring immediate investigation.

```
